In [0]:
spark.sql("USE inventory_ai")

forecast_df  = spark.table("gold_demand_forecast")
inventory_df = spark.table("silver_inventory")
supplier_df  = spark.table("silver_supplier_lead_time")


In [0]:
risk_df = (
    forecast_df
    .join(inventory_df, on="product_family", how="left")
    .join(supplier_df, on="product_family", how="left")
)


In [0]:
from pyspark.sql.functions import col

risk_df = risk_df.withColumn(
    "expected_demand_during_lead_time",
    col("prediction") * col("lead_time_days")
)


In [0]:
risk_df = risk_df.withColumn(
    "stock_gap",
    col("current_stock") - col("expected_demand_during_lead_time")
)


In [0]:
from pyspark.sql.functions import when

risk_df = risk_df.withColumn(
    "stockout_risk_score",
    when(col("stock_gap") < 0, 1.0)
    .otherwise(
        1 - (col("stock_gap") / (col("current_stock") + 1))
    )
)


In [0]:
risk_df = risk_df.withColumn(
    "stockout_risk_level",
    when(col("stockout_risk_score") >= 0.8, "HIGH")
    .when(col("stockout_risk_score") >= 0.4, "MEDIUM")
    .otherwise("LOW")
)


In [0]:
f = forecast_df.alias("f")
i = inventory_df.alias("i")
s = supplier_df.alias("s")


In [0]:
from pyspark.sql.functions import col, when

# Alias tables to avoid ambiguity
f = spark.table("inventory_ai.gold_demand_forecast").alias("f")
i = spark.table("inventory_ai.silver_inventory").alias("i")
s = spark.table("inventory_ai.silver_supplier_lead_time").alias("s")

# Join forecast with inventory and supplier data
risk_df = (
    f.join(
        i,
        (f.store_id == i.store_id) &
        (f.product_family == i.product_family),
        "left"
    )
    .join(
        s,
        f.product_family == s.product_family,
        "left"
    )
)

# Calculate expected demand during lead time
risk_df = risk_df.withColumn(
    "expected_demand_during_lead_time",
    col("f.prediction") * col("s.lead_time_days")
)

# Calculate stock gap
risk_df = risk_df.withColumn(
    "stock_gap",
    col("i.current_stock") - col("expected_demand_during_lead_time")
)

# Stock-out risk score
risk_df = risk_df.withColumn(
    "stockout_risk_score",
    when(col("stock_gap") < 0, 1.0)
    .otherwise(
        1 - (col("stock_gap") / (col("i.current_stock") + 1))
    )
)

# Risk level classification
risk_df = risk_df.withColumn(
    "stockout_risk_level",
    when(col("stockout_risk_score") >= 0.8, "HIGH")
    .when(col("stockout_risk_score") >= 0.4, "MEDIUM")
    .otherwise("LOW")
)

# Final Gold table
risk_df.select(
    col("f.date").alias("date"),
    col("f.store_id").alias("store_id"),
    col("f.product_family").alias("product_family"),
    col("f.prediction").alias("predicted_daily_demand"),
    col("s.lead_time_days"),
    col("i.current_stock"),
    col("expected_demand_during_lead_time"),
    col("stock_gap"),
    col("stockout_risk_score"),
    col("stockout_risk_level")
).write \
 .format("delta") \
 .mode("overwrite") \
 .saveAsTable("inventory_ai.gold_stockout_risk")


In [0]:
%sql
show tables

In [0]:
spark.sql("""
SELECT store_id, product_family, stockout_risk_level, stockout_risk_score
FROM inventory_ai.gold_stockout_risk
ORDER BY stockout_risk_score DESC
LIMIT 10
""").show()
